<a href="https://colab.research.google.com/github/TWAPZCODE/Quant-Research/blob/main/Volatility_Regime_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
       accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

In [31]:
# Data parameters
ticker = "QQQ"
start = "2010-01-01"
end = "2025-01-01"

# Target parameters
vol_lookahead = 10    # days forward to measure realized volatility over
vol_lookback_windows = [5, 10, 20] # backward-looking vol features (days)

In [32]:
data = yf.download(ticker, start=start, end=end, auto_adjust=True)
data.columns = data.columns.get_level_values(0) if isinstance(data.columns, pd.MultiIndex) else data.columns
data = data.dropna()
data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,40.246574,40.307268,40.116525,40.168547,62822800
2010-01-05,40.246574,40.315937,40.021153,40.220565,62935600
2010-01-06,40.003807,40.359281,39.943117,40.229231,96033000
2010-01-07,40.029812,40.116515,39.813060,40.064493,77094100
2010-01-08,40.359280,40.359280,39.821735,39.943116,88886600


In [33]:
df = data.copy()

# Daily log returns
df["log_ret"] = np.log(df["Close"] / df["Close"].shift(1))

# Backward-looking realized volatility features (annualised)
for w in vol_lookback_windows:
      df[f"realized_vol_{w}d"] = df["log_ret"].rolling(w).std() * np.sqrt(252)

# Momentum / returns
df["ret_5d"] = df["Close"].pct_change(5)
df["ret_10d"] = df["Close"].pct_change(10)
df["ret_20d"] = df["Close"].pct_change(20)

# Average True Range (ATR) - volatility of price range
high_low = df["High"] - df["Low"]
high_close = (df["High"] - df["Close"].shift()).abs()
low_close = (df["Low"] - df["Close"].shift()).abs()
true_ranges = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df["atr_14"] = true_ranges.rolling(14).mean()

# RSI (14-day)
delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# Volume change
df["volume_change_5d"] = df["Volume"].pct_change(5)

df.head()
df.tail()

Price,Close,High,Low,Open,Volume,log_ret,realized_vol_5d,realized_vol_10d,realized_vol_20d,ret_5d,ret_10d,ret_20d,atr_14,rsi_14,volume_change_5d
Date,,,,,,,,,,,,,,,
2024-12-24,526.043762,526.133063,520.316382,520.951667,17558200,0.013469,0.327603,0.254987,0.194624,-0.009304,0.019621,0.047819,7.921226,54.902115,-0.385628
2024-12-26,525.686279,527.314174,522.420614,524.415770,19090500,-0.000680,0.120019,0.239311,0.194561,0.027076,0.001008,0.041515,8.085306,55.689442,-0.651017
2024-12-27,518.698364,522.559632,514.033084,522.122881,33839600,-0.013382,0.171563,0.247014,0.199189,0.017956,-0.005883,0.035825,8.570118,48.036106,-0.271355
2024-12-30,511.799683,515.521971,508.047618,511.700446,34584000,-0.013389,0.199311,0.249953,0.204321,-0.004278,-0.026556,0.013147,8.958549,46.343592,-0.428647
2024-12-31,507.452087,513.834533,506.489254,513.080200,29117000,-0.008531,0.179535,0.232143,0.203086,-0.022262,-0.048527,-0.006279,9.042917,44.902078,-0.018731


In [ ]:
future_log_ret